#Oldest way of v1

but each way need to buy credits

In [ ]:
import tweepy
import csv

consumer_key = "---"
consumer_secret = "---"
access_key = "---"
access_secret = "---"

In [ ]:
def get_all_tweets(screen_name):
    auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
    auth.set_access_token(access_key, access_secret)
    api = tweepy.API(auth)


    alltweets = []
    new_tweets = api.user_timeline(screen_name = screen_name,count=200)
    alltweets.extend(new_tweets)
    oldest = alltweets[-1].id - 1
    while len(new_tweets) > 0:
        print(f"getting tweets before {oldest}")
        new_tweets = api.user_timeline(screen_name = screen_name,count=200,max_id=oldest)
        alltweets.extend(new_tweets)
        oldest = alltweets[-1].id - 1
        print(f"...{len(alltweets)} tweets downloaded so far")

    outtweets = [[tweet.id_str, tweet.created_at, tweet.text] for tweet in alltweets]
    with open(f'new_{screen_name}_tweets.csv', 'w') as f:
        writer = csv.writer(f)
        writer.writerow(["id","created_at","text"])
        try :
            writer.writerows(outtweets)
        except :
            pass

    pass


get_all_tweets("elonmusk")

getting tweets before 1408257232765083653
...25 tweets downloaded so far
getting tweets before 1408247691537223680
...25 tweets downloaded so far


#Now v2

In [7]:
import tweepy
import csv

# ==========================================
# Your Bearer Token
# ==========================================
BEARER_TOKEN = "YOUR_BEARER_TOKEN"

# Create Client
client = tweepy.Client(
    bearer_token=BEARER_TOKEN,
    wait_on_rate_limit=True
)


def get_all_tweets(username):
    # --------------------------------------
    # Get User ID from username
    # --------------------------------------
    user = client.get_user(
        username=username,
        user_fields=["id", "name", "username"]
    )

    if user.data is None:
        print("User not found.")
        return

    user_id = user.data.id

    print(f"User ID: {user_id}")
    print(f"Downloading tweets from @{username}...\n")

    all_tweets = []
    next_token = None

    while True:

        response = client.get_users_tweets(
            id=user_id,
            max_results=100,
            pagination_token=next_token,
            tweet_fields=[
                "created_at",
                "text",
                "public_metrics",
                "lang"
            ]
        )

        if response.data is None:
            break

        all_tweets.extend(response.data)

        print(f"Downloaded {len(all_tweets)} tweets")

        next_token = response.meta.get("next_token")

        if next_token is None:
            break

    print(f"\nTotal Tweets: {len(all_tweets)}")

    filename = f"{username}_tweets.csv"

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "id",
            "created_at",
            "text",
            "likes",
            "retweets",
            "replies",
            "quotes",
            "language"
        ])

        for tweet in all_tweets:
            writer.writerow([
                tweet.id,
                tweet.created_at,
                tweet.text,
                tweet.public_metrics["like_count"],
                tweet.public_metrics["retweet_count"],
                tweet.public_metrics["reply_count"],
                tweet.public_metrics["quote_count"],
                tweet.lang
            ])

    print(f"\nSaved to {filename}")


if __name__ == "__main__":
    get_all_tweets("elonmusk")

HTTPException: 402 Payment Required
credits depleted